# Air Quality Analysis and Prediction (India 2010–2023)
### B.E. Computer Science Semester Project
**Official Problem Statement:**
> *"Develop a data science and machine learning solution that analyzes environmental data to identify air quality patterns and predict air quality levels."*

---
### Notebook Workflow:
1. **Environment Setup & Configuration**
2. **Data Ingestion & CPCB Preprocessing**
3. **Exploratory Data Analysis (EDA) & Pollution Dynamics**
4. **Feature Engineering & Leak-Free Time Split**
5. **Model Benchmarking & Selection (5 Architectures)**
6. **Model Explainability & Error Diagnostics**
7. **Inference & Health Advisory Demonstration**

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from src.config import TARGET_COL, POLLUTANT_COLS, MET_COLS
from src.preprocessor import load_processed
from src.features import build_features, time_aware_split
from src.predictor import predict_from_simple_inputs, compute_aqi_from_pm25_scalar
print('All modules loaded successfully.')

## 1. Load Processed Clean Dataset
We load the 6.49M-row preprocessed parquet dataset covering 133 CPCB monitoring stations.

In [ ]:
df = load_processed()
print(f'Total records: {len(df):,}')
print(f'Stations: {df["station_id"].nunique()} across {df["city"].nunique()} cities')
df.head(3)

## 2. Exploratory Data Analysis & Empirical Patterns
Let us inspect city-wide PM2.5 averages and diurnal rush-hour cycles.

In [ ]:
city_summary = df.groupby('city', observed=True)[TARGET_COL].agg(['mean', 'median', 'count']).reset_index()
city_summary.columns = ['City', 'Mean PM2.5', 'Median PM2.5', 'Records']
city_summary.sort_values('Mean PM2.5', ascending=False)

## 3. Feature Engineering & Chronological Partitioning
We create lag features, rolling statistics, and perform strict time-aware splitting.

In [ ]:
from src.config import TRAIN_END_DATE, VAL_END_DATE
print(f'Train bound: <= {TRAIN_END_DATE}')
print(f'Val bound:   {TRAIN_END_DATE} to {VAL_END_DATE}')
print(f'Test bound:  > {VAL_END_DATE}')

## 4. Model Benchmarking Summary
Results from our 5-model regression evaluation on the test partition.

In [ ]:
from app.pages_models import _load_metrics_df
metrics_table = _load_metrics_df()
metrics_table

## 5. Live Prediction Engine Demonstration
Testing 1-hour ahead PM2.5 prediction and CPCB health advisory for a winter episode in Delhi.

In [ ]:
sample_forecast = predict_from_simple_inputs(
    pm25_current=165.0,
    temperature=15.0,
    relative_humidity=80.0,
    wind_speed=0.9,
    hour=20,
    month=11
)
print(f"Predicted PM2.5: {sample_forecast['predicted_pm25']} ug/m3")
print(f"Predicted NAQI:  {sample_forecast['predicted_aqi']} ({sample_forecast['aqi_category']})")
print(f"Advisory:        {sample_forecast['advisory']}")
print('Precautions:')
for p in sample_forecast['precautions']:
    print(f'  - {p}')